In [4]:
import text_lloom.workbench as wb
import os
import sys
import csv
import argparse
import pandas as pd
from pathlib import Path
# from vllm import LLM, SamplingParams
from tqdm import tqdm

In [7]:
# Test write permission
test_file_path = os.path.join(os.environ['TRANSFORMERS_CACHE'], 'text.txt')
try:
    with open(test_file_path, 'w') as f:
        f.write('This is a test.')
    print('Write successful!')
except Exception as e:
    print('Error writing to directory: ', e)

Write successful!


In [1]:
import text_lloom.workbench as wb
from text_lloom.llm import Model, EmbedModel
from openai import OpenAI

VLLM_BASE_URL = "http://localhost:8001"  # match your sbatch server

def setup_llm_fn(api_key):
    return OpenAI(api_key=api_key, base_url=VLLM_BASE_URL)

def setup_embed_fn(api_key):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("BAAI/bge-large-en-v1.5")

async def call_llm_fn(model, prompt):
    """LLooM call function: vLLM OpenAI-compatible chat completion."""
    res = model.client.chat.completions.create(
        model=model.name,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a helpful assistant who helps with identifying patterns in text examples."},
            {"role": "user", "content": prompt},
        ],
    )
    return res.choices[0].message.content, (0, 0)

def call_embed_fn(model, text_arr):
    embeddings = model.client.encode(text_arr).tolist()
    return embeddings, (0, 0)

In [2]:
api_key = "EMPTY"  # vLLM ignores this but LLooM requires a value

l = wb.lloom(
    df=advice_df,
    id_col="narrative_idx",
    text_col="advice_text",
    distill_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn,
                         name="Qwen/Qwen2.5-14B-Instruct", cost=[0, 0],
                         rate_limit=(300, 10), context_window=32768, api_key=api_key),
    cluster_model=EmbedModel(setup_fn=setup_embed_fn, fn=call_embed_fn,
                              name="bge-large-en-v1.5", cost=0,
                              batch_size=2048, api_key=api_key),
    synth_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn,
                       name="Qwen/Qwen2.5-14B-Instruct", cost=[0, 0],
                       rate_limit=(20, 10), context_window=32768, api_key=api_key),
    score_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn,
                       name="Qwen/Qwen2.5-14B-Instruct", cost=[0, 0],
                       rate_limit=(300, 10), context_window=32768, api_key=api_key),
)

NameError: name 'advice_df' is not defined

In [7]:
gemma_advice_t1 = pd.read_csv('../../6_Explicit-Implicit-Bias/Results/AdviceGeneration/FullResults-May5/gemma_advice_t1.csv')
gemma_advice_t2 = pd.read_csv('../../6_Explicit-Implicit-Bias/Results/AdviceGeneration/FullResults-May5/gemma_advice_t2.csv')

In [9]:
gemma_advice_t1.info()

<class 'pandas.DataFrame'>
RangeIndex: 14793 entries, 0 to 14792
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   narrative_idx     14793 non-null  int64
 1   narrative         14793 non-null  str  
 2   nli_labels        14793 non-null  str  
 3   prompt_variant    14793 non-null  str  
 4   model             14793 non-null  str  
 5   response_t1       14793 non-null  str  
 6   n_tokens_t1       14793 non-null  int64
 7   finish_reason_t1  14793 non-null  str  
 8   batch_idx         14793 non-null  int64
dtypes: int64(3), str(6)
memory usage: 1.0 MB


In [10]:
gemma_advice_t2.info()

<class 'pandas.DataFrame'>
RangeIndex: 723672 entries, 0 to 723671
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   narrative_idx     723672 non-null  int64
 1   myth_type         334848 non-null  str  
 2   myth_pair         388824 non-null  str  
 3   frame             723672 non-null  str  
 4   dose              723672 non-null  int64
 5   condition         723672 non-null  str  
 6   myth_statement    723672 non-null  str  
 7   prompt_variant    723672 non-null  str  
 8   model             723672 non-null  str  
 9   response_t2       723672 non-null  str  
 10  n_tokens_t2       723672 non-null  int64
 11  finish_reason_t2  723672 non-null  str  
dtypes: int64(3), str(9)
memory usage: 66.3 MB
